In [2]:
from pennylane import numpy as np
import pennylane as qml
import itertools
from itertools import combinations
from scipy.optimize import minimize
from math import comb
import qaoa_aux 

In [3]:
def U_B(beta,edge_list):
    for edge_idx, (u, v) in enumerate(edge_list):
        theta = beta[edge_idx]
        qml.S(wires=u)
        qml.SingleExcitationMinus(-2 * theta,wires=[u, v])
        qml.adjoint(qml.S)(wires=u)

In [4]:
def U_B_standard(beta, edge_list):
    for u, v in edge_list:
        qml.S(wires=u)
        qml.SingleExcitationMinus(-2 * beta, wires=[u, v])
        qml.adjoint(qml.S)(wires=u)

In [5]:
def U_C(gamma,cost_terms):
    # we have term index + cost terms so all the even numbered ZZ's
    for term_idx, (_, wires, coeff) in enumerate(cost_terms):
        theta = 2 * gamma[term_idx] * coeff
        # example for 4 consectuve z's
        #qml.PauliRot(theta, "ZZZZ", wires=[0,1,2,3])
        qml.PauliRot(theta,"Z" * len(wires),wires=list(wires))

In [6]:
def U_C_MC(gamma,edge_list):
    edge_idx = 0
    for u, v in edge_list: 
        
        qml.CNOT(wires=(u, v))
        qml.RZ(gamma[edge_idx], wires=v)
        qml.CNOT(wires=(u, v))
        edge_idx += 1 


In [7]:
def objective(gammas, betas, circuit,hyperedge_list, cost_terms, weights):
    expectation_values = circuit(gammas, betas)

    # Store weighted expected cut value for each hyperedge
    edge_values = []
    for edge_idx, edge in enumerate(hyperedge_list):
        k = len(edge)
        # Weighted constant term for this hyperedge
        edge_value = weights[edge_idx] * (1 - 1 / (2 ** (k - 1)))

        # Add the Pauli-Z terms belonging to this hyperedge
        for term_idx, (source_edge_idx, wires, coeff) in enumerate(cost_terms):
            if source_edge_idx == edge_idx:
                edge_value += coeff * expectation_values[term_idx]
        edge_values.append(edge_value)

    # Negative because Adam minimizes
    return qml.math.max(qml.math.stack(edge_values))



    

In [8]:
def run_qaoa(
    hyperedge_list,
    n,
    weights,
    runtype,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.01,
    max_trials=100
):
    n_wires = n
    assert len(weights) == len(hyperedge_list)

    # Build mixer edges and cost terms
    cost_edge_list = qaoa_aux.clique_expansion(hyperedge_list)
    mixer_edge_list = qaoa_aux.connected_clique_expansion(hyperedge_list, n)
    cost_terms = qaoa_aux.build_cost_terms(hyperedge_list,weights)

    # Balanced initial state
    target_weight = (n_wires + 1) // 2
    initial_state = qaoa_aux.make_dicke_state(n_wires, target_weight)

    dev = qml.device("lightning.qubit", wires=n_wires)
    
    @qml.qnode(dev, diff_method="adjoint",device_vjp=True)
    def aon_circuit(gammas, betas):
        qml.StatePrep(initial_state,wires=range(n_wires))

        for layer in range(num_layers):
            U_C(gammas[:, layer],cost_terms)
            U_B(betas[:, layer],mixer_edge_list)

        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables

    @qml.qnode(dev, diff_method="adjoint",device_vjp=True)
    def mc_circuit(gammas,betas):
        qml.StatePrep(initial_state,wires=range(n_wires))
        for layer in range(num_layers):
            U_C_MC(gammas[:,layer],cost_edge_list)
            U_B(betas[:, layer],mixer_edge_list)
        observables = []

        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire)for wire in wires])
            observables.append(qml.expval(z_term))
        return observables

    @qml.qnode(dev, diff_method="adjoint",device_vjp=True)
    def standard_circuit_aon(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))    
        for layer in range(num_layers):
            # Multi-angle AoN cost
            U_C(gammas[:, layer], cost_terms)
            # Single-angle mixer
            U_B_standard(betas[layer], mixer_edge_list)
    
        observables = []
    
        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire) for wire in wires])
            observables.append(qml.expval(z_term))
    
        return observables

    @qml.qnode(dev, diff_method="adjoint",device_vjp=True)
    def standard_circuit_mc(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))    
        for layer in range(num_layers):
            # single-angle AoN cost
            U_C_MC(gammas[:,layer],cost_edge_list)
            # Single-angle mixer
            U_B_standard(betas[layer], mixer_edge_list)
    
        observables = []
    
        for _, wires, _ in cost_terms:
            z_term = qml.prod(*[qml.PauliZ(wire) for wire in wires])
            observables.append(qml.expval(z_term))
    
        return observables

    

    if runtype == "aon":
        circuit = aon_circuit
        n_gamma = len(cost_terms)
        single_beta = False
    
    elif runtype == "mc":
        circuit = mc_circuit
        n_gamma = len(cost_edge_list)
        single_beta = False
    
    elif runtype == "standard_aon":
        circuit = standard_circuit_aon
        n_gamma = len(cost_terms)
        single_beta = True
    elif runtype == "standard_mc":
        circuit = standard_circuit_mc
        n_gamma = len(cost_edge_list)
        single_beta = True
    else:
        raise ValueError("runtype must be 'aon', 'mc','standard_aon', 'standard_mc'")

    # Random parameter initialization
    rng = np.random.default_rng(seed)
    
    gamma_params = np.array(scaling_constant * rng.random((n_gamma, num_layers)),requires_grad=True)
    # Single-angle or multi-angle mixer parameters
    if single_beta:
        beta_shape = (num_layers,)
    else:
        beta_shape = (len(mixer_edge_list), num_layers)
    
    beta_params = np.array(scaling_constant * rng.random(beta_shape),requires_grad=True)
    
    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_values = []
    print("Gamma shape:", gamma_params.shape)
    print("Beta shape:", beta_params.shape)
    print("Total parameters:", gamma_params.size + beta_params.size)


    # Define optimization objective
    def loss(g, b):
        return objective(
            g,
            b,
            circuit,
            hyperedge_list,
            cost_terms,
            weights
        )
    
    initial_value = loss(gamma_params, beta_params)
    objective_values.append(float(initial_value))
    
    print(f"Initial objective = {initial_value:.6f} | Runtype = {runtype}")
    
    # Optimization
    for trial in range(max_trials):
    
        gamma_params, beta_params = opt.step(
            loss,
            gamma_params,
            beta_params
        )
    
        current_value = loss(gamma_params, beta_params)
        objective_values.append(float(current_value))
    
        if trial % 10 == 0:
            print(
                f"Iteration {trial:3d} | "
                f"objective = {current_value:.6f}"
            )
    
    return {
        "gammas": gamma_params,
        "betas": beta_params,
        "objective_values": objective_values,
        "circuit": circuit,
        "cost_edge_list": cost_edge_list,
        "mixer_edge_list": mixer_edge_list,
        "cost_terms": cost_terms,
        "n_wires": n_wires
    }

        

In [17]:
P = np.load("../data/poisson_16_12_3/P/03.npy")
w= np.load("../data/poisson_16_12_3/w/03.npy")

hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)

In [19]:

result = run_qaoa(
    hyperedge_list,
    weights=w,
    runtype="aon",
    n=12,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.05,
    max_trials=300
)

Initial objective = 0.939304 | Runtype = aon
Iteration   0 | objective = 0.939659
Iteration  50 | objective = 0.826701
Iteration 100 | objective = 0.823764
Iteration 150 | objective = 0.817693
Iteration 200 | objective = 0.810769
Iteration 250 | objective = 0.810449


In [20]:

result = run_qaoa(
    hyperedge_list,
    weights=w,
    runtype="mc",
    n=12,
    num_layers=1,
    seed=42,
    scaling_constant=0.1,
    step_size=0.05,
    max_trials=300
)

Initial objective = 0.946669 | Runtype = mc
Iteration   0 | objective = 0.945554
Iteration  50 | objective = 0.838498
Iteration 100 | objective = 0.822963
Iteration 150 | objective = 0.814445
Iteration 200 | objective = 0.816199
Iteration 250 | objective = 0.818714


In [21]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_3/P"
folder_w = "../data/poisson_16_12_3/w"

p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_aon = []
results_mc=[]

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p} | Runtype=AoN")

        result_aon = run_qaoa(hyperedge_list,weights=w,runtype="aon",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_aon["objective_values"][-1])

        initial_objective = float(result_aon["objective_values"][0])

        results_aon.append({
            "dataset": "poisson_16_12_3",
            "instance": instance,
            "runtype": "AoN",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_aon["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_aon["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })
        print(f"\nRunning instance {instance} | p={p} | Runtype=AoN")

        result_mc = run_qaoa(hyperedge_list,weights=w,runtype="mc",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_mc["objective_values"][-1])

        initial_objective = float(result_mc["objective_values"][0])

        results_mc.append({
            "dataset": "poisson_16_12_3",
            "instance": instance,
            "runtype": "MC",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_mc["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_mc["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })


        
df_aon = pd.DataFrame(results_aon)
df_mc = pd.DataFrame(results_mc)
csv_path_aon = "poisson_16_12_3_fair_min_bisection_aon.csv"
csv_path_mc = "poisson_16_12_3_fair_min_bisection_mc.csv"

df_aon.to_csv(csv_path_aon,index=False)
df_mc.to_csv(csv_path_mc,index=False)

print(f"saved results to: {csv_path_aon} | {csv_path_mc}")


Running instance 01 | p=3 | Runtype=AoN
Initial objective = 1.555111 | Runtype = aon
Iteration   0 | objective = 1.467707
Iteration  50 | objective = 0.965749
Iteration 100 | objective = 0.914168
Iteration 150 | objective = 0.881455
Iteration 200 | objective = 0.851244
Iteration 250 | objective = 0.842599

Running instance 01 | p=3 | Runtype=AoN
Initial objective = 1.645531 | Runtype = mc
Iteration   0 | objective = 1.577698
Iteration  50 | objective = 0.973945
Iteration 100 | objective = 0.893663
Iteration 150 | objective = 0.865395
Iteration 200 | objective = 0.847858
Iteration 250 | objective = 0.829311

Running instance 02 | p=3 | Runtype=AoN
Initial objective = 0.985837 | Runtype = aon
Iteration   0 | objective = 0.982232
Iteration  50 | objective = 0.828244
Iteration 100 | objective = 0.794915
Iteration 150 | objective = 0.782849
Iteration 200 | objective = 0.777775
Iteration 250 | objective = 0.771368

Running instance 02 | p=3 | Runtype=AoN
Initial objective = 0.988100 | Runty

In [22]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_4/P"
folder_w = "../data/poisson_16_12_4/w"

p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_aon = []
results_mc=[]

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p} | Runtype=AoN")

        result_aon = run_qaoa(hyperedge_list,weights=w,runtype="aon",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_aon["objective_values"][-1])

        initial_objective = float(result_aon["objective_values"][0])

        results_aon.append({
            "dataset": "poisson_16_12_3",
            "instance": instance,
            "runtype": "AoN",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_aon["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_aon["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })
        # print(f"\nRunning instance {instance} | p={p} | Runtype=MC")

        # result_mc = run_qaoa(hyperedge_list,weights=w,runtype="mc",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        # final_objective = float(result_mc["objective_values"][-1])

        # initial_objective = float(result_mc["objective_values"][0])

        # results_mc.append({
        #     "dataset": "poisson_16_12_3",
        #     "instance": instance,
        #     "runtype": "MC",
        #     "p": p,
        #     "n_vertices": n,
        #     "n_hyperedges": len(hyperedge_list),
        #     "gammas": np.array2string(np.asarray(result_mc["gammas"]),separator=","),
        #     "betas": np.array2string(np.asarray(result_mc["betas"]),separator=","),
        #     "initial_objective": initial_objective,
        #     "final_objective": final_objective 
        # }
        


        
df_aon = pd.DataFrame(results_aon)
# df_mc = pd.DataFrame(results_mc)
csv_path_aon = "poisson_16_12_4_fair_min_bisection_aon.csv"
# csv_path_mc = "poisson_16_12_4_fair_min_bisection_mc.csv"

df_aon.to_csv(csv_path_aon,index=False)
df_mc.to_csv(csv_path_mc,index=False)

print(f"saved results to: {csv_path_aon} | {csv_path_mc}")


Running instance 01 | p=3 | Runtype=AoN
Initial objective = 0.984587 | Runtype = aon
Iteration   0 | objective = 0.982648
Iteration  50 | objective = 0.909572
Iteration 100 | objective = 0.887671
Iteration 150 | objective = 0.879771
Iteration 200 | objective = 0.874343
Iteration 250 | objective = 0.867586

Running instance 02 | p=3 | Runtype=AoN
Initial objective = 1.000000 | Runtype = aon
Iteration   0 | objective = 1.000000
Iteration  50 | objective = 1.000000
Iteration 100 | objective = 1.000000
Iteration 150 | objective = 1.000000
Iteration 200 | objective = 1.000000
Iteration 250 | objective = 1.000000

Running instance 03 | p=3 | Runtype=AoN
Initial objective = 0.998057 | Runtype = aon
Iteration   0 | objective = 0.997951
Iteration  50 | objective = 0.922396
Iteration 100 | objective = 0.885476
Iteration 150 | objective = 0.869618
Iteration 200 | objective = 0.858846
Iteration 250 | objective = 0.862062

Running instance 04 | p=3 | Runtype=AoN
Initial objective = 1.000000 | Runt

In [8]:
import os
import glob
import pandas as pd
folder_P = "../data/poisson_16_12_5/P"
folder_w = "../data/poisson_16_12_5/w"

p_files = sorted(glob.glob(f"{folder_P}/*.npy"))[:20]

results_aon = []
results_mc=[]

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    P = np.load(p_file)
    w_file = os.path.join(folder_w,f"{instance}.npy")
    w = np.load(w_file)
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p} | Runtype=AoN")

        result_aon = run_qaoa(hyperedge_list,weights=w,runtype="aon",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_aon["objective_values"][-1])

        initial_objective = float(result_aon["objective_values"][0])

        results_aon.append({
            "dataset": "poisson_16_12_5",
            "instance": instance,
            "runtype": "AoN",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_aon["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_aon["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })
        # print(f"\nRunning instance {instance} | p={p} | Runtype=MC")

        # result_mc = run_qaoa(hyperedge_list,weights=w,runtype="mc",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        # final_objective = float(result_mc["objective_values"][-1])

        # initial_objective = float(result_mc["objective_values"][0])

        # results_mc.append({
        #     "dataset": "poisson_16_12_3",
        #     "instance": instance,
        #     "runtype": "MC",
        #     "p": p,
        #     "n_vertices": n,
        #     "n_hyperedges": len(hyperedge_list),
        #     "gammas": np.array2string(np.asarray(result_mc["gammas"]),separator=","),
        #     "betas": np.array2string(np.asarray(result_mc["betas"]),separator=","),
        #     "initial_objective": initial_objective,
        #     "final_objective": final_objective 
        # }
        


        
df_aon = pd.DataFrame(results_aon)
# df_mc = pd.DataFrame(results_mc)
csv_path_aon = "poisson_16_12_5_fair_min_bisection_aon.csv"
# csv_path_mc = "poisson_16_12_4_fair_min_bisection_mc.csv"

df_aon.to_csv(csv_path_aon,index=False)
# df_mc.to_csv(csv_path_mc,index=False)

print(f"saved results to: {csv_path_aon}")


Running instance 01 | p=3 | Runtype=AoN
Initial objective = 1.000000 | Runtype = aon
Iteration   0 | objective = 1.000000
Iteration  50 | objective = 1.000000
Iteration 100 | objective = 1.000000
Iteration 150 | objective = 1.000000
Iteration 200 | objective = 1.000000
Iteration 250 | objective = 1.000000

Running instance 02 | p=3 | Runtype=AoN
Initial objective = 1.000000 | Runtype = aon
Iteration   0 | objective = 1.000000
Iteration  50 | objective = 1.000000
Iteration 100 | objective = 1.000000
Iteration 150 | objective = 1.000000
Iteration 200 | objective = 1.000000
Iteration 250 | objective = 1.000000

Running instance 03 | p=3 | Runtype=AoN
Initial objective = 1.000000 | Runtype = aon
Iteration   0 | objective = 1.000000
Iteration  50 | objective = 1.000000
Iteration 100 | objective = 1.000000
Iteration 150 | objective = 1.000000
Iteration 200 | objective = 1.000000
Iteration 250 | objective = 1.000000

Running instance 04 | p=3 | Runtype=AoN
Initial objective = 1.000000 | Runt

IOStream.flush timed out


Iteration 250 | objective = 1.000000
saved results to: poisson_16_12_5_fair_min_bisection_aon.csv


In [ ]:
import os
import glob
import pandas as pd
folder_P = "/Users/stevehalley/Code/Github/QuantumHypergraphPartitioning/data/email-Enron/numpy_15"
def get_edge_list(P):
    edges=[]
    for idx in range(P.shape[1]):
        #indices that are not 0
        edge = tuple(np.flatnonzero(P[:,idx]))
        edges.append(edge)
    return edges

p_files = sorted(glob.glob(f"{folder_P}/*.npy"))

results_aon = []
results_mc=[]

for p_file in p_files:

    instance = os.path.splitext(os.path.basename(p_file))[0]
    # Load hypergraph
    P = np.load(p_file)
    edges = get_edge_list(P)
    w = np.ones(len(edges))
    hyperedge_list = qaoa_aux.matrix_to_hyperedges(P)
    n = P.shape[0]

    #circuit depth
    for p in [3]:

        print(f"\nRunning instance {instance} | p={p} | Runtype=Standard,Single-angle Mixer")

        result_aon = run_qaoa(hyperedge_list,weights=w,runtype="standard_mc",n=n,num_layers=p,seed=42,scaling_constant=5e-2,step_size=0.01,max_trials=300)
        final_objective = float(result_aon["objective_values"][-1])

        initial_objective = float(result_aon["objective_values"][0])

        results_aon.append({
            "dataset": "enron",
            "instance": instance,
            "runtype": "AoN",
            "p": p,
            "n_vertices": n,
            "n_hyperedges": len(hyperedge_list),
            "gammas": np.array2string(np.asarray(result_aon["gammas"]),separator=","),
            "betas": np.array2string(np.asarray(result_aon["betas"]),separator=","),
            "initial_objective": initial_objective,
            "final_objective": final_objective 
        })

        
df_aon = pd.DataFrame(results_aon)
# df_mc = pd.DataFrame(results_mc)
csv_path_aon = "enron_fair_min_bisection_aon_single_angle_mixer.csv"
# csv_path_mc = "poisson_16_12_4_fair_min_bisection_mc.csv"

df_aon.to_csv(csv_path_aon,index=False)
# df_mc.to_csv(csv_path_mc,index=False)

print(f"saved results to: {csv_path_aon}")


Running instance email-Enron_1 | p=3 | Runtype=Standard,Single-angle Mixer
Gamma shape: (18, 3)
Beta shape: (3,)
Total parameters: 57
Initial objective = 0.935177 | Runtype = standard_mc
Iteration   0 | objective = 0.931009
Iteration  10 | objective = 0.894109
Iteration  20 | objective = 0.843605
Iteration  30 | objective = 0.791539
Iteration  40 | objective = 0.737331
Iteration  50 | objective = 0.671918
Iteration  60 | objective = 0.617647
Iteration  70 | objective = 0.590011
Iteration  80 | objective = 0.563671
Iteration  90 | objective = 0.550424
Iteration 100 | objective = 0.540922
Iteration 110 | objective = 0.532262
Iteration 120 | objective = 0.527131
Iteration 130 | objective = 0.519844
Iteration 140 | objective = 0.519849
Iteration 150 | objective = 0.512679
Iteration 160 | objective = 0.511552
Iteration 170 | objective = 0.505438
Iteration 180 | objective = 0.502622
Iteration 190 | objective = 0.499806
Iteration 200 | objective = 0.497995
Iteration 210 | objective = 0.49383